In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn


SEED = 42
N_VOLTAGE_ANCHORS = 128

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

l = 300e-6
wb = 1e-6
wa = 1e-6
he = 60e-6
E = 170e9
nu = 0.28
epsilon_0 = 8.854e-12
V_MAX = 8.0

E_prime = E / (1.0 - nu**2)
I = he * wb**3 / 12.0
alpha = 6.0 * (wa / wb) ** 2
gamma_xy = (l / wa) ** 2
beta_max = epsilon_0 * he * V_MAX**2 * l**4 / (2.0 * E_prime * I * wa**3)

class SirenLayer(nn.Module):
    def __init__(self, in_features, out_features, is_first=False, omega_0=30.0):
        super().__init__()
        self.omega_0 = omega_0
        self.linear = nn.Linear(in_features, out_features)

        with torch.no_grad():
            if is_first:
                bound = 1.0 / in_features
            else:
                bound = np.sqrt(6.0 / in_features) / omega_0
            self.linear.weight.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


class SingleCoupledPINN(nn.Module):


    def __init__(self, hidden_dim=180, num_layers=5, omega_0=30.0):
        super().__init__()
        layers = [SirenLayer(3, hidden_dim, is_first=True, omega_0=omega_0)]
        for _ in range(num_layers - 1):
            layers.append(SirenLayer(hidden_dim, hidden_dim, omega_0=omega_0))
        self.net = nn.Sequential(*layers)
        self.final = nn.Linear(hidden_dim, 2)

        with torch.no_grad():
            bound = np.sqrt(6.0 / hidden_dim) / omega_0
            self.final.weight.uniform_(-bound, bound)
            self.final.bias.zero_()

    def raw_outputs(self, xi, eta, v_star):
        net_in = torch.cat([xi, eta, v_star], dim=1)
        return self.final(self.net(net_in))

    def mechanical(self, xi, eta, v_star):

        eta_ref = torch.full_like(eta, ETA_REFERENCE)
        raw_w = self.raw_outputs(xi, eta_ref, v_star)[:, 0:1]
        spatial_shape = xi**2 * (1.0 - xi) ** 2
        return v_star**2 * spatial_shape * raw_w

    def electric(self, xi, eta, v_star):
        s_xi = xi**2 * (3.0 - 2.0 * xi)
        raw_phi = self.raw_outputs(s_xi, eta, v_star)[:, 1:2]
        return (1.0 - eta) + eta * (1.0 - eta) * raw_phi

    def fields(self, xi, eta, v_star):
        return self.mechanical(xi, eta, v_star), self.electric(xi, eta, v_star)

    def forward(self, xi, eta, v_star):
        return self.fields(xi, eta, v_star)

    def mechanical_with_x_derivatives(self, xi, eta, v_star):

        eta = torch.full_like(eta, ETA_REFERENCE)
        z0 = torch.cat([xi, eta, v_star], dim=1)
        z1 = torch.cat(
            [torch.ones_like(xi), torch.zeros_like(eta), torch.zeros_like(v_star)],
            dim=1,
        )
        z2 = torch.zeros_like(z0)
        z3 = torch.zeros_like(z0)
        z4 = torch.zeros_like(z0)

        for layer in self.net:
            omega = layer.omega_0
            weight = layer.linear.weight

            a0 = omega * layer.linear(z0)
            a1 = omega * torch.nn.functional.linear(z1, weight, None)
            a2 = omega * torch.nn.functional.linear(z2, weight, None)
            a3 = omega * torch.nn.functional.linear(z3, weight, None)
            a4 = omega * torch.nn.functional.linear(z4, weight, None)

            sin_a = torch.sin(a0)
            cos_a = torch.cos(a0)

            z0 = sin_a
            z1 = cos_a * a1
            z2 = cos_a * a2 - sin_a * a1**2
            z3 = cos_a * a3 - 3.0 * sin_a * a1 * a2 - cos_a * a1**3
            z4 = (
                cos_a * a4
                - 4.0 * sin_a * a1 * a3
                - 3.0 * sin_a * a2**2
                - 6.0 * cos_a * a1**2 * a2
                + sin_a * a1**4
            )


        w_weight = self.final.weight[0:1, :]
        w_bias = self.final.bias[0:1]
        raw0 = torch.nn.functional.linear(z0, w_weight, w_bias)
        raw1 = torch.nn.functional.linear(z1, w_weight, None)
        raw2 = torch.nn.functional.linear(z2, w_weight, None)
        raw3 = torch.nn.functional.linear(z3, w_weight, None)
        raw4 = torch.nn.functional.linear(z4, w_weight, None)

        g0 = xi**2 * (1.0 - xi) ** 2
        g1 = 2.0 * xi - 6.0 * xi**2 + 4.0 * xi**3
        g2 = 2.0 - 12.0 * xi + 12.0 * xi**2
        g3 = -12.0 + 24.0 * xi
        g4 = torch.full_like(xi, 24.0)
        scale = v_star**2

        w0 = scale * g0 * raw0
        w1 = scale * (g1 * raw0 + g0 * raw1)
        w2 = scale * (g2 * raw0 + 2.0 * g1 * raw1 + g0 * raw2)
        w3 = scale * (
            g3 * raw0 + 3.0 * g2 * raw1 + 3.0 * g1 * raw2 + g0 * raw3
        )
        w4 = scale * (
            g4 * raw0
            + 4.0 * g3 * raw1
            + 6.0 * g2 * raw2
            + 4.0 * g1 * raw3
            + g0 * raw4
        )
        return w0, w1, w2, w3, w4


ETA_REFERENCE = 0.5


def compute_T_bar_exact(model, v_star_value, n_quad=201):
    if n_quad % 2 == 0:
        n_quad += 1

    xi_q = torch.linspace(0.001, 0.999, n_quad, device=device).view(-1, 1)
    eta_q = torch.full_like(xi_q, ETA_REFERENCE)
    v_q = torch.full_like(xi_q, float(v_star_value))
    _, w_xi, _, _, _ = model.mechanical_with_x_derivatives(xi_q, eta_q, v_q)

    f = w_xi.squeeze(1) ** 2
    h_step = (0.999 - 0.001) / (n_quad - 1)
    weights = torch.ones(n_quad, device=device)
    weights[1:-1:2] = 4.0
    weights[2:-1:2] = 2.0
    return (h_step / 3.0) * torch.sum(weights * f)


def compute_T_bar_batch(model, v_star_m, n_quad=201):
    if n_quad % 2 == 0:
        n_quad += 1

    v_flat = v_star_m.squeeze(1)
    unique_v, inv_idx = torch.unique(v_flat.detach(), return_inverse=True)
    num_v = unique_v.shape[0]

    xi_lin = torch.linspace(0.001, 0.999, n_quad, device=device)
    xi_big = xi_lin.repeat(num_v).unsqueeze(1)
    eta_big = torch.full_like(xi_big, ETA_REFERENCE)
    v_big = unique_v.repeat_interleave(n_quad).unsqueeze(1)

    _, w_xi_big, _, _, _ = model.mechanical_with_x_derivatives(
        xi_big, eta_big, v_big
    )
    f_big = w_xi_big.squeeze(1).pow(2).view(num_v, n_quad)

    h_step = (0.999 - 0.001) / (n_quad - 1)
    weights = torch.ones(n_quad, device=device)
    weights[1:-1:2] = 4.0
    weights[2:-1:2] = 2.0
    weights *= h_step / 3.0

    t_unique = torch.sum(f_big * weights.unsqueeze(0), dim=1)
    return t_unique[inv_idx].unsqueeze(1)


def compute_mech_loss(model, xi_m, v_star_m, alpha_scale=1.0):
    eta_ref = torch.full_like(xi_m, ETA_REFERENCE)
    w, _, w_xx, _, w_xxxx = model.mechanical_with_x_derivatives(
        xi_m, eta_ref, v_star_m
    )

    h = w + 1.0
    h_sq = h**2
    t_bar = compute_T_bar_batch(model, v_star_m, n_quad=201)

    eta_top = torch.ones_like(xi_m, requires_grad=True)
    phi_top = model.electric(xi_m, eta_top, v_star_m)
    phi_eta_top = torch.autograd.grad(
        phi_top,
        eta_top,
        grad_outputs=torch.ones_like(phi_top),
        create_graph=True,
    )[0]

    beta_dynamic = beta_max * v_star_m**2
    residual_raw = (
        h_sq * w_xxxx
        - h_sq * alpha * alpha_scale * t_bar * w_xx
        + beta_dynamic * phi_eta_top**2
    )
    scaling = 1.0 / (v_star_m**2 + 1e-3)
    return torch.mean((residual_raw * scaling) ** 2)


def compute_elec_loss(model, xi, eta, v_star):

    xi = xi.detach().requires_grad_(True)
    eta = eta.detach().requires_grad_(True)

    eta_ref = torch.full_like(xi, ETA_REFERENCE)
    w = model.mechanical(xi, eta_ref, v_star)
    h = torch.clamp(w + 1.0, min=0.1)
    phi = model.electric(xi, eta, v_star)

    phi_xi = torch.autograd.grad(
        phi, xi, torch.ones_like(phi), create_graph=True
    )[0]
    phi_eta = torch.autograd.grad(
        phi, eta, torch.ones_like(phi), create_graph=True
    )[0]
    phi_y = phi_eta / h

    w_xi = torch.autograd.grad(w, xi, torch.ones_like(w), create_graph=True)[0]
    phi_x = phi_xi - eta * w_xi * phi_y

    phi_yy = (
        torch.autograd.grad(phi_y, eta, torch.ones_like(phi_y), create_graph=True)[0]
        / h
    )
    phi_x_xi = torch.autograd.grad(
        phi_x, xi, torch.ones_like(phi_x), create_graph=True
    )[0]
    phi_x_eta = torch.autograd.grad(
        phi_x, eta, torch.ones_like(phi_x), create_graph=True
    )[0]
    phi_xx = phi_x_xi - eta * w_xi * phi_x_eta / h

    residual = (1.0 / gamma_xy) * phi_xx + phi_yy
    return torch.mean(residual**2)


def compute_pde_residual(model, voltage, n_pts=201):
    model.eval()
    xi = torch.linspace(0.01, 0.99, n_pts, device=device).view(-1, 1)
    v_star = torch.full_like(xi, voltage / V_MAX)
    eta_ref = torch.full_like(xi, ETA_REFERENCE)

    w, _, w_xx, _, w_xxxx = model.mechanical_with_x_derivatives(
        xi, eta_ref, v_star
    )
    t_bar = compute_T_bar_exact(model, voltage / V_MAX, n_quad=201)
    h = w + 1.0

    eta_top = torch.ones_like(xi, requires_grad=True)
    phi_top = model.electric(xi, eta_top, v_star)
    phi_eta_top = torch.autograd.grad(
        phi_top, eta_top, torch.ones_like(phi_top), create_graph=True
    )[0]

    beta_dynamic = beta_max * v_star**2
    residual = (
        w_xxxx
        - alpha * t_bar * w_xx
        + beta_dynamic * (phi_eta_top / h) ** 2
    )
    result = torch.mean(torch.abs(residual)).item()
    model.train()
    return result


def _uniform_voltage_anchors(v_star_max, n_anchors=N_VOLTAGE_ANCHORS):

    edges = torch.linspace(0.0, v_star_max, n_anchors + 1, device=device)
    anchors = edges[:-1] + torch.rand(n_anchors, device=device) * (
        edges[1:] - edges[:-1]
    )
    return anchors.view(-1, 1)


def get_samples(n_mech, n_elec, v_star_max):
    xi_m = torch.rand(n_mech, 1, device=device) * 0.98 + 0.01
    anchors = _uniform_voltage_anchors(v_star_max)
    repeats = (n_mech + anchors.shape[0] - 1) // anchors.shape[0]
    v_star_m = anchors.repeat(repeats, 1)[:n_mech]
    v_star_m = v_star_m[torch.randperm(n_mech, device=device)]

    xi_e = torch.rand(n_elec, 1, device=device) * 0.98 + 0.01
    eta_e = torch.rand(n_elec, 1, device=device)
    v_star_e = torch.rand(n_elec, 1, device=device) * v_star_max
    return xi_m, v_star_m, xi_e, eta_e, v_star_e


def evaluate_model(model, voltage):
    model.eval()
    xi = torch.linspace(0.0, 1.0, 100, device=device).view(-1, 1)
    eta_ref = torch.full_like(xi, ETA_REFERENCE)
    v_star = torch.full_like(xi, voltage / V_MAX)
    with torch.no_grad():
        w = model.mechanical(xi, eta_ref, v_star)
        max_deflection = w.min().item() * wa * 1e6
        min_gap = (w + 1.0).min().item()
    model.train()
    return max_deflection, min_gap


class TrainingController:


    def __init__(self, initial_voltage=2.0):
        self.current_voltage = initial_voltage
        self.voltage_step = 0.5
        self.voltage_history = []
        self.residual_history = []
        self.deflection_history = []
        self.loss_mech_history = []
        self.loss_elec_history = []
        self.voltage_advance_epochs = []
        self.loss_history = []
        self.gap_history = []
        self.epochs_at_current_voltage = 0
        self.max_epochs_per_voltage = 20000
        self.local_gap_history = []
        self.local_mech_loss_history = []
        self.local_elec_loss_history = []
        self.local_res_samples = []
        self.RESIDUAL_SAMPLE_INTERVAL = 1000

    def get_min_epochs(self):
        history = self.local_gap_history if self.local_gap_history else self.gap_history
        if not history:
            return 4000
        gap = history[-1]
        if gap < 0.45:
            return 20000
        if gap < 0.55:
            return 16000
        if gap < 0.65:
            return 14000
        if gap < 0.75:
            return 12000
        if gap < 0.85:
            return 10000
        return 8000

    def get_dynamic_threshold(self):
        history = self.local_gap_history if self.local_gap_history else self.gap_history
        if not history:
            return 0.02
        gap = history[-1]
        if gap < 0.35:
            return 0.005
        if gap < 0.45:
            return 0.01
        if gap < 0.60:
            return 0.02
        if gap < 0.75:
            return 0.03
        return 0.04

    @staticmethod
    def _window_change(values, window):
        recent = float(np.mean(values[-window:]))
        previous = float(np.mean(values[-2 * window:-window]))
        change = abs(previous - recent) / (abs(previous) + 1e-10)
        return recent, change

    def is_converged(self):
        if self.epochs_at_current_voltage < self.get_min_epochs():
            return False
        window = 2000
        if len(self.local_mech_loss_history) < 2 * window:
            return False
        if len(self.local_elec_loss_history) < 2 * window:
            return False

        threshold = self.get_dynamic_threshold()
        recent_mech, mech_change = self._window_change(
            self.local_mech_loss_history, window
        )
        recent_elec, elec_change = self._window_change(
            self.local_elec_loss_history, window
        )
        if recent_mech > 0.06 or recent_elec > 0.06:
            return False

        residual_stable = False
        if len(self.local_res_samples) >= 4:
            recent_res = float(np.mean(self.local_res_samples[-2:]))
            previous_res = float(np.mean(self.local_res_samples[-4:-2]))
            residual_change = abs(previous_res - recent_res) / (
                abs(previous_res) + 1e-10
            )
            residual_stable = residual_change < threshold

        return (
            mech_change < threshold
            and elec_change < threshold
            and residual_stable
        )

    def check_residual_stop(self, residual):
        log_residual = np.log10(residual + 1e-15)
        print(f"    log10(res)={log_residual:.3f}")
        if log_residual > 1.2:
            return True, f"log10(res)={log_residual:.3f} > 1.2"
        return False, ""

    def try_expand_voltage(self, model, alpha_scale):
        del alpha_scale
        old_voltage = self.current_voltage
        self.voltage_advance_epochs.append((len(self.loss_history), old_voltage))
        deflection, gap = evaluate_model(model, old_voltage)
        self.deflection_history.append(deflection)
        self.voltage_history.append(old_voltage)
        residual = compute_pde_residual(model, old_voltage)
        self.residual_history.append(residual)
        print(
            f"    V={old_voltage:.2f} V, w={deflection:.6f} um, "
            f"log10(res_end)={np.log10(residual + 1e-15):.3f}"
        )
        should_stop, reason = self.check_residual_stop(residual)
        if should_stop:
            return old_voltage, True, reason
        if gap < 0.45:
            self.voltage_step = 0.05
        elif gap < 0.55:
            self.voltage_step = 0.08
        elif gap < 0.65:
            self.voltage_step = 0.10
        elif gap < 0.75:
            self.voltage_step = 0.15
        else:
            self.voltage_step = 0.30
        self.current_voltage += self.voltage_step
        self.epochs_at_current_voltage = 0
        self.local_gap_history = []
        self.local_mech_loss_history = []
        self.local_elec_loss_history = []
        self.local_res_samples = []
        print(
            f">>> voltage continuation: {old_voltage:.2f} V -> "
            f"{self.current_voltage:.2f} V (min_ep={self.get_min_epochs()})"
        )
        return self.current_voltage, False, ""

    def update(self, mech_loss, elec_loss, min_gap):
        mech_loss = float(mech_loss)
        elec_loss = float(elec_loss)
        self.loss_mech_history.append((len(self.loss_history), mech_loss))
        self.loss_elec_history.append((len(self.loss_history), elec_loss))
        self.loss_history.append(0.8 * mech_loss + 0.2 * elec_loss)
        self.gap_history.append(float(min_gap))
        self.local_gap_history.append(float(min_gap))
        self.local_mech_loss_history.append(mech_loss)
        self.local_elec_loss_history.append(elec_loss)
        self.epochs_at_current_voltage += 1
        if self.is_converged():
            return "EXPAND"
        if self.epochs_at_current_voltage >= self.max_epochs_per_voltage:
            return "EXPAND"
        return "CONTINUE"

    def get_v_star_max(self):
        return self.current_voltage / V_MAX

    def get_lr_factor(self):
        history = self.local_gap_history if self.local_gap_history else self.gap_history
        if not history:
            return 1.0
        gap = history[-1]
        if gap < 0.45:
            return 0.02
        if gap < 0.55:
            return 0.05
        if gap < 0.65:
            return 0.10
        if gap < 0.75:
            return 0.30
        if gap < 0.85:
            return 0.50
        return 1.0

    def state_dict(self):
        return dict(self.__dict__)

    def load_state_dict(self, state):
        for key, value in state.items():
            setattr(self, key, value)

    def get_summary(self):
        return {
            "final_voltage": self.current_voltage,
            "total_epochs": len(self.loss_history),
            "voltage_data": list(
                zip(
                    self.voltage_history,
                    self.residual_history,
                    self.deflection_history,
                )
            ),
        }

    def summary(self):
        return self.get_summary()


def pcgrad_shared_step(loss_mech, loss_elec, model, optimizer, mech_share=0.80):

    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    grad_mech_raw = torch.autograd.grad(
        loss_mech, parameters, retain_graph=True, allow_unused=True
    )
    grad_elec_raw = torch.autograd.grad(
        loss_elec, parameters, retain_graph=False, allow_unused=True
    )
    grad_mech = [
        torch.zeros_like(parameter) if gradient is None else gradient
        for parameter, gradient in zip(parameters, grad_mech_raw)
    ]
    grad_elec = [
        torch.zeros_like(parameter) if gradient is None else gradient
        for parameter, gradient in zip(parameters, grad_elec_raw)
    ]

    dot = sum(torch.sum(gm * ge) for gm, ge in zip(grad_mech, grad_elec))
    norm_mech_sq = sum(torch.sum(gm * gm) for gm in grad_mech)
    norm_elec_sq = sum(torch.sum(ge * ge) for ge in grad_elec)
    cosine = dot / torch.sqrt(norm_mech_sq * norm_elec_sq + 1e-30)
    conflict = bool(dot.item() < 0.0)

    if conflict:
        original_mech = grad_mech
        original_elec = grad_elec
        grad_mech = [
            gm - dot / (norm_elec_sq + 1e-30) * ge
            for gm, ge in zip(original_mech, original_elec)
        ]
        grad_elec = [
            ge - dot / (norm_mech_sq + 1e-30) * gm
            for gm, ge in zip(original_mech, original_elec)
        ]

    projected_mech_norm = torch.sqrt(
        sum(torch.sum(gradient * gradient) for gradient in grad_mech) + 1e-30
    )
    projected_elec_norm = torch.sqrt(
        sum(torch.sum(gradient * gradient) for gradient in grad_elec) + 1e-30
    )
    elec_share = 1.0 - mech_share

    optimizer.zero_grad(set_to_none=True)
    for parameter, gm, ge in zip(parameters, grad_mech, grad_elec):
        parameter.grad = (
            mech_share * gm / projected_mech_norm
            + elec_share * ge / projected_elec_norm
        ).detach()
    torch.nn.utils.clip_grad_norm_(parameters, max_norm=1.0)
    optimizer.step()
    return (
        float(torch.sqrt(norm_mech_sq + 1e-30).item()),
        float(torch.sqrt(norm_elec_sq + 1e-30).item()),
        float(cosine.item()),
        conflict,
    )


def atomic_save_checkpoint(path, epoch, model, optimizer, controller, loss_weights):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "controller_state": controller.state_dict(),
            "loss_weights": loss_weights,
            "summary": controller.summary(),
            "training_variant": "fair single shared SIREN; normalized PCGrad 4:1; separate Phase-4 task losses",
        },
        temporary,
    )
    os.replace(temporary, path)


def train():
    hidden_dim = 180
    model = SingleCoupledPINN(hidden_dim=hidden_dim, num_layers=5, omega_0=30.0).to(device)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    print(f"Single-network trainable parameters: {parameter_count:,}")

    base_lr = 1e-4
    optimizer = torch.optim.Adam(model.parameters(), lr=base_lr)
    controller = TrainingController(initial_voltage=2.0)
    checkpoint_path = "single_shared_siren.pth"
    max_epochs = 500000
    checkpoint_interval = 5000
    start_epoch = 0
    task_settings = {
        "comparison_role": "single-vs-dual architecture ablation",
        "single_network": "one fully shared hidden trunk with two output neurons",
        "input": "(xi, eta, v_star)",
        "mechanical_eta_reference": ETA_REFERENCE,
        "optimizer": "normalized PCGrad joint optimization with 4:1 task emphasis",
        "mechanical_gradient_share": 0.80,
        "electrical_gradient_share": 0.20,
        "voltage_sampling": "128 unbiased equal-width stratified anchors",
        "mechanical_points": 3000,
        "electrical_points": 5000,
        "simpson_points": 201,
        "mechanical_envelope_scale": 1.0,
        "alpha_scale": 1.0,
        "seed": SEED,
        "hidden_dim": hidden_dim,
        "num_layers": 5,
        "parameter_count": parameter_count,
    }

    if os.path.exists(checkpoint_path):
        if os.path.getsize(checkpoint_path) < 1024:
            raise EOFError(
                f"Existing checkpoint {checkpoint_path} is empty/incomplete. "
                "Delete or rename only that incomplete file before retrying."
            )
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint["model_state"], strict=True)
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        controller.load_state_dict(checkpoint["controller_state"])
        start_epoch = int(checkpoint.get("epoch", -1)) + 1
        print(f"Resumed from epoch {start_epoch}, V={controller.current_voltage:.2f} V")

    start_time = time.time()
    epoch = start_epoch
    _, cached_gap = evaluate_model(model, controller.current_voltage)
    last_grad_info = (float("nan"), float("nan"), float("nan"), False)

    while epoch < max_epochs:
        v_star_max = controller.get_v_star_max()
        xi_m, v_star_m, xi_e, eta_e, v_star_e = get_samples(
            3000, 5000, v_star_max
        )
        alpha_scale = 1.0

        lr_factor = controller.get_lr_factor()
        for group in optimizer.param_groups:
            group["lr"] = base_lr * lr_factor

        loss_mech = compute_mech_loss(model, xi_m, v_star_m, alpha_scale)
        loss_elec = compute_elec_loss(model, xi_e, eta_e, v_star_e)
        last_grad_info = pcgrad_shared_step(
            loss_mech, loss_elec, model, optimizer, mech_share=0.80
        )

        if epoch % controller.RESIDUAL_SAMPLE_INTERVAL == 0 and epoch > start_epoch:
            residual = compute_pde_residual(model, controller.current_voltage)
            controller.local_res_samples.append(residual)

        _, cached_gap = evaluate_model(model, controller.current_voltage)
        action = controller.update(loss_mech.item(), loss_elec.item(), cached_gap)

        if action == "EXPAND":
            result_voltage, stop_flag, stop_reason = controller.try_expand_voltage(
                model, alpha_scale
            )
            atomic_save_checkpoint(
                checkpoint_path, epoch, model, optimizer, controller, task_settings
            )
            if stop_flag:
                print(f"Training stopped: {stop_reason}")
                break
            predicted_w, predicted_gap = evaluate_model(model, result_voltage)
            cached_gap = predicted_gap
            print(
                f"    prediction: w={predicted_w:.6f} um, gap={predicted_gap:.4f}"
            )
            if predicted_gap < 0.25:
                print(f"gap={predicted_gap:.4f} < 0.25")
                break

        if epoch % controller.RESIDUAL_SAMPLE_INTERVAL == 0:
            displacement, cached_gap = evaluate_model(
                model, controller.current_voltage
            )
            residual_text = (
                f"{controller.local_res_samples[-1]:.3e}"
                if controller.local_res_samples
                else "N/A"
            )
            g_mech, g_elec, cosine, conflict = last_grad_info
            print(
                f"Epoch {epoch:6d} | V={controller.current_voltage:.2f} V | "
                f"w={displacement:.6f} um | gap={cached_gap:.4f} | "
                f"Lm={loss_mech.item():.3e} | Le={loss_elec.item():.3e} | "
                f"res={residual_text} | gm/ge={g_mech/(g_elec+1e-30):.2e} | "
                f"cos={cosine:+.3f} | projected={conflict} | "
                f"lr_factor={lr_factor:.3f} | "
                f"local_ep={controller.epochs_at_current_voltage}"
                f"[min={controller.get_min_epochs()}]"
            )

        if epoch > start_epoch and epoch % checkpoint_interval == 0:
            atomic_save_checkpoint(
                checkpoint_path, epoch, model, optimizer, controller, task_settings
            )
        epoch += 1

    atomic_save_checkpoint(
        checkpoint_path, epoch, model, optimizer, controller, task_settings
    )
    print(f"Finished. Elapsed time: {(time.time() - start_time) / 60.0:.1f} min")


if __name__ == "__main__":
    train()
